# THE MOMENT — Zero-Config Kaggle Run

**Upload this kit as a Kaggle Dataset → New Notebook with GPU T4 x2 → Run All.**

Everything is packaged: 52 reference face photos, script, score prompts, configs.
The only download at runtime is the Wan2.2-S2V Q8 GGUF model (~14GB, cached after first run).
TTS models (Kokoro, IndicParler) auto-download from HuggingFace on first use.

## What you get
- `output/THE_MOMENT.mp4` — H.264, 48fps, cinema-grade, -16 LUFS audio
- `output/THE_MOMENT_MASTER_4444.mov` — ProRes 4:4:4 10-bit (if KIT_MASTER=1)
- Per-shot checkpoints → resume if session drops
- Identity auto-retry → same face across all shots

In [ ]:
# === SETUP ===
import os, sys, yaml, json, subprocess, shutil, urllib.request, zipfile, glob
os.chdir('/kaggle/working')

# Copy kit files from Dataset to working (Kaggle Datasets are read-only)
import glob as _glob
kit_srcs = _glob.glob('/kaggle/input/*')
kit_src = next((s for s in kit_srcs if os.path.isdir(os.path.join(s, 'src'))), None)
if kit_src:
    for item in os.listdir(kit_src):
        if item.startswith('.'): continue
        s = os.path.join(kit_src, item)
        d = os.path.join('/kaggle/working', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

print('Kit root:', os.getcwd())
print('Files:', len(os.listdir()))

In [ ]:
# === DOWNLOAD HUNYUANVIDEO-1.5 480P DISTILLED (8 STEPS, Q4_K_S) ===
cfg = yaml.safe_load(open('config.yaml'))
models_dir = cfg['paths']['models']
os.makedirs(models_dir, exist_ok=True)

model_url = 'https://huggingface.co/jayn7/HunyuanVideo-1.5_I2V_480p-GGUF/resolve/main/distilled/1.5.0-hunyuan-480p-i2v-distilled-q4_k_s.gguf'
model_path = os.path.join(models_dir, '1.5.0-hunyuan-480p-i2v-distilled-q4_k_s.gguf')

if not os.path.exists(model_path):
    print('Downloading HunyuanVideo-1.5 480p Distilled Q4_K_S (~5GB)...')
    print('This takes 5-15 min on first run. Persistent storage keeps it cached.')
    urllib.request.urlretrieve(model_url, model_path)
    print(f'Download complete: {os.path.getsize(model_path)/1e9:.1f} GB')
else:
    print(f'Model cached: {os.path.getsize(model_path)/1e9:.1f} GB')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config.yaml'
cfg['active_profile'] = 'kaggle_2xt4'
yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)
print('Profile: kaggle_2xt4 (FREE, 2x T4) - HunyuanVideo-1.5 8-step distilled')

In [ ]:
# === SMOKE TEST (static checks, no GPU) ===
env = {**os.environ, 'COMFY_MOCK': '1', 'POST_MODE': 'cpu', 'KIT_FAST_POST': '1'}
r = subprocess.run([sys.executable, 'src/selftest.py', '--fast'], env=env)
print(f'Smoke test: {"GREEN" if r.returncode==0 else "FAILED"}')
if r.returncode != 0:
    raise SystemExit('Fix issues above')

In [ ]:
# === STEP 1: COST METER (1 sample shot, then HOLDS) ===
os.environ.setdefault('BUDGET_USD', '4')
!python3 src/filmforge.py
print('\nReview projection above. Proceed to STEP 2 if OK.')

In [ ]:
# === STEP 2: FULL RENDER ===
# Uncomment next line for ProRes 4:4:4 master:
# os.environ['KIT_MASTER'] = '1'
os.environ['CONFIRM_FULL_RUN'] = '1'
!python3 src/filmforge.py
print('\n===== RENDER COMPLETE =====')

In [ ]:
# === STEP 3: QC + PACKAGE ===
!python3 src/qc.py

out_dir, targets = 'output', ['THE_MOMENT.mp4', 'THE_MOMENT_MASTER_4444.mov']
existing = [f for f in targets if os.path.exists(os.path.join(out_dir, f))]
zip_path = '/kaggle/working/THE_MOMENT_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in existing:
        z.write(os.path.join(out_dir, f), f)
print(f'Delivery zip: {zip_path}')
print('Download from Kaggle Output tab → Download All.')